# Encoder-Decoder Algorithm Architecture

In [14]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, num_layers,
                           dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        _, (hidden, cell) = self.rnn(embedded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, num_layers,
                           dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell


class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        pass

if __name__ == "__main__":
    INPUT_DIM = 10000
    OUTPUT_DIM = 10000
    EMB_DIM = 256
    HIDDEN_DIM = 512
    NUM_LAYERS = 2
    DROPOUT = 0.5

    encoder = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
    decoder = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
    model = EncoderDecoder(encoder, decoder)

    print("=== Encoder-Decoder Model Architecture ===")
    print(model)
    print("\nTotal trainable parameters:",
          sum(p.numel() for p in model.parameters() if p.requires_grad))

=== Encoder-Decoder Model Architecture ===
EncoderDecoder(
  (encoder): Encoder(
    (embedding): Embedding(10000, 256)
    (rnn): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(10000, 256)
    (rnn): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=10000, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

Total trainable parameters: 17606416


# SEQ-2-SEQ Practical Example


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_data = [
    ("hello world", "bonjour le monde"),
    ("how are you", "comment allez vous"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("i love you", "je t aime"),
    ("what is your name", "comment t appelles tu"),
    ("i am fine", "je vais bien"),
    ("see you later", "a plus tard"),
]

def build_vocab(sentences):
    vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
    for sent in sentences:
        for word in sent.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

src_vocab = build_vocab([pair[0] for pair in train_data])
tgt_vocab = build_vocab([pair[1] for pair in train_data])

src_vocab_size = len(src_vocab)
tgt_vocab_size = len(tgt_vocab)
tgt_inv_vocab = {idx: word for word, idx in tgt_vocab.items()}

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = tgt.shape[1]
        tgt_len = tgt.shape[0]
        tgt_vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(tgt_len, batch_size, tgt_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        input = tgt[0, :]

        for t in range(1, tgt_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = tgt[t] if teacher_force else top1

        return outputs

# Hyperparameters
EMB_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5

encoder = Encoder(src_vocab_size, EMB_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
decoder = Decoder(tgt_vocab_size, EMB_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Model created successfully!")

Model created successfully!


In [10]:

# Hyperparameters
EMB_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5
LEARNING_RATE = 0.001
N_EPOCHS = 200
teacher_forcing_ratio = 0.8

encoder = Encoder(src_vocab_size, EMB_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
decoder = Decoder(tgt_vocab_size, EMB_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=0)
output = model(src_tensor, tgt_tensor, teacher_forcing_ratio=teacher_forcing_ratio)

# ====================== TRAINING ======================
print("Starting Training...\n")

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0

    for src_sent, tgt_sent in train_data:
        src_tensor = sentence_to_tensor(src_sent, src_vocab).unsqueeze(1)
        tgt_tensor = sentence_to_tensor(tgt_sent, tgt_vocab).unsqueeze(1)

        optimizer.zero_grad()
        output = model(src_tensor, tgt_tensor)

        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        tgt = tgt_tensor[1:].view(-1)

        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{N_EPOCHS}], Loss: {epoch_loss/len(train_data):.4f}")

print("\nTraining Completed!\n")

# ====================== INFERENCE ======================
def translate(sentence):
    model.eval()
    with torch.no_grad():
        src_tensor = sentence_to_tensor(sentence, src_vocab).unsqueeze(1)
        hidden, cell = model.encoder(src_tensor)

        input = torch.tensor([tgt_vocab['<sos>']], dtype=torch.long).to(device)
        output_sentence = []

        for _ in range(20):  # max length
            output, hidden, cell = model.decoder(input, hidden, cell)
            top1 = output.argmax(1).item()
            if top1 == tgt_vocab['<eos>']:
                break
            output_sentence.append(tgt_inv_vocab[top1])
            input = torch.tensor([top1], dtype=torch.long).to(device)

    return ' '.join(output_sentence)

# Test the model
test_sentences = [
    "hello world",
    "thank you",
    "how are you",
    "good morning",
    "i love you"
]

print("=== TRANSLATION RESULTS ===\n")
for sent in test_sentences:
    translation = translate(sent)
    print(f"English : {sent}")
    print(f"French  : {translation}")
    print("-" * 40)

Starting Training...

Epoch [10/200], Loss: 0.7550
Epoch [20/200], Loss: 0.0071
Epoch [30/200], Loss: 0.0023
Epoch [40/200], Loss: 0.0014
Epoch [50/200], Loss: 0.0009
Epoch [60/200], Loss: 0.0007
Epoch [70/200], Loss: 0.0005
Epoch [80/200], Loss: 0.0004
Epoch [90/200], Loss: 0.0003
Epoch [100/200], Loss: 0.0003
Epoch [110/200], Loss: 0.0002
Epoch [120/200], Loss: 0.0002
Epoch [130/200], Loss: 0.0002
Epoch [140/200], Loss: 0.0002
Epoch [150/200], Loss: 0.0001
Epoch [160/200], Loss: 0.0001
Epoch [170/200], Loss: 0.0001
Epoch [180/200], Loss: 0.0001
Epoch [190/200], Loss: 0.0001
Epoch [200/200], Loss: 0.0001

Training Completed!

=== TRANSLATION RESULTS ===

English : hello world
French  : bonjour le monde
----------------------------------------
English : thank you
French  : merci
----------------------------------------
English : how are you
French  : comment allez vous
----------------------------------------
English : good morning
French  : bonjour
------------------------------------

In [11]:
print("=== Interactive English to French Translator ===")
print("Type 'quit' or 'exit' to stop\n")

while True:
    user_input = input("English: ").strip()

    if user_input.lower() in ['quit', 'exit', 'q']:
        print("Goodbye!")
        break

    if user_input == "":
        continue

    translation = translate(user_input)
    print(f"French : {translation}")
    print("-" * 50)

=== Interactive English to French Translator ===
Type 'quit' or 'exit' to stop

English: i love you
French : je t aime
--------------------------------------------------
English: how are you
French : comment allez vous
--------------------------------------------------
English: what is your namw
French : comment t appelles tu
--------------------------------------------------
English: exit
Goodbye!
